# 02 Equivalent Image Generation

## Visualizing Classifier Equivalence

Two images are **equivalent** under a classifier if they produce identical output logits.
For example, any feature vector can be decomposed into a *principal* component (what drives
the prediction) and a *null* component (what the classifier is blind to). Strip the null
component away, and the prediction is unchanged. The image, however, may look very
different.

This notebook makes that difference visible. Starting from a single input image, we:

1. Extract penultimate-layer features and decompose the classifier head via SVD
2. Remove the null-space component to produce an *equivalent* feature vector
3. Translate both feature vectors into the Karlo vision-language embedding space
4. Generate images from each embedding using **UnCLIP** (Kakao Karlo)

The result is a pair of images: a reconstruction of the original scene, and its
null-space-free equivalent — same prediction, different visual content. The gap between
them is exactly what the classifier has learned to ignore.

---

The notebook is structured in two parts:

- **Part A** — manual, step-by-step pipeline: each stage is explicit and inspectable.
- **Part B** — the same result in a few lines using the `SingleImageAnalyzer` high-level
  API


In [ ]:
#---------------------------- setup install colab --------------------------------#
!git clone https://github.com/harel314/SING-analyzing-semantic-invariants-classifiers.git
%cd SING-analyzing-semantic-invariants-classifiers
%pip install .

## LOCAL USERS
simply run `uv sync` and continue from here

In [ ]:
#---------------------------- gpu check --------------------------------#
!nvidia-smi

In [ ]:
#---------------------------- imports and config --------------------------------#
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

from sing.core.projectors import compute_projectors_from_weight, principal_component
from sing.generation.generate import generate_seed_set
from sing.generation.unclip_wrapper import KakaoUnclipWrapper
from sing.models.registry import load_model
from sing.translators.registry import load_default_translator

try:
    import google.colab
    repo_root = Path(".").resolve()
except ImportError:
    repo_root = Path("..").resolve()

image_path = repo_root / "samples" / "border_collie_n02106166.jpeg"
if not image_path.exists():
    raise FileNotFoundError(f"Set image_path to an existing image. Missing: {image_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

---
## Part A — Step-by-step manual pipeline

In [ ]:
#---------------------------- load model and translator --------------------------------#
model_name = "resnet"

loaded_model = load_model(model_name=model_name, device=device)
loaded_translator = load_default_translator(
    translators_root=repo_root / "translators",
    registry_path=repo_root / "translators" / "registry.yaml",
    model_name=loaded_model.name,
    device=device,
)
print(f"model={loaded_model.name}")
print(f"translator={loaded_translator.metadata.translator_name} architecture={loaded_translator.metadata.architecture}")

### Step 1 — Extract Features and Project Out the Null Space

We pass the image through the backbone and capture the penultimate-layer activations.
Those are the representation the classifier head actually operates on.

Then we decompose the classifier weight matrix with SVD. This gives us two orthogonal
bases: `v_principal` (the directions the classifier *uses*) and `v_null` (the directions
it *ignores*). The rank of the classifier head determines the size of each subspace.
for a 1000-class head, the principal subspace has at most 1000 directions, while the null
space accounts for the rest.

The equivalent feature vector is computed by projecting out the null component:

$$\tilde{f} = f - f \cdot V_\text{null} \cdot V_\text{null}^\top$$

`f` and `f̃` produce identical logits. Everything that differs between them lives
entirely in the null space.


In [ ]:
#---------------------------- extract features and project --------------------------------#
image = Image.open(image_path).convert("RGB")
input_tensor = loaded_model.preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    features = loaded_model.wrapper.extract_features(input_tensor)
    classifier_weight = loaded_model.wrapper.classifier_weight.detach().to(device)
    projectors = compute_projectors_from_weight(classifier_weight)
    principal_features = principal_component(features, projectors.v_null)

print(f"features={tuple(features.shape)}")
print(f"projector_rank={projectors.rank} v_null={tuple(projectors.v_null.shape)} v_principal={tuple(projectors.v_principal.shape)}")
print(f"principal_features={tuple(principal_features.shape)}")

### Step 2 — Translate to Vision-Language Embedding Space

The classifier features live in a backbone-specific space and not directly comparable to
text or generatable by an image synthesis model. The translator bridges that gap: a
lightweight network trained to map penultimate-layer features to Kakao/Karlo CLIP
embeddings (dim 768).

Both the original and equivalent features are translated independently. The resulting
embeddings sit in a shared vision-language space where angular distance is meaningful
and where an UnCLIP model can condition on them to generate images.


In [ ]:
#---------------------------- translate to embedding space --------------------------------#
with torch.no_grad():
    translated_original = loaded_translator.model(features)
    translated_principal = loaded_translator.model(principal_features)

print(f"translated_original={tuple(translated_original.shape)}")
print(f"translated_principal={tuple(translated_principal.shape)}")

### Step 3 - Generate Images with UnCLIP

With both embeddings in Karlo's CLIP space, we can now condition an image generation
model on them. We use **Kakao Karlo**, an UnCLIP model that generates images directly
from CLIP embeddings rather than text prompts.

For each random seed, two images are generated side by side:
- one conditioned on the **original** embedding (reconstruction of the input scene)
- one conditioned on the **principal** embedding (the null-space-free equivalent)

Running multiple seeds lets you separate the effect of the null-space removal from
stochastic variation in the generator. Content that consistently disappears across seeds
is attributable to the null space, not sampling noise.


In [ ]:
#---------------------------- load unclip and generate --------------------------------#
seeds = [42, 1337]
output_dir = repo_root / "outputs" / "notebook_generation_manual"

unclip = KakaoUnclipWrapper(
    device=device,
    torch_dtype=KakaoUnclipWrapper.default_dtype(device),
)

results = generate_seed_set(
    wrapper=unclip,
    image=image,
    principal_embedding=translated_principal.detach(),
    seeds=seeds,
    output_dir=output_dir,
)

print(f"generated {len(results)} seed pair(s) -> {output_dir}")

### Step 4 - Visual Comparison

Each row corresponds to one random seed. The left column is a reconstruction of the
original image conditioned on the full feature embedding. The right column is its
equivalent: same prediction, null space removed.

Look at what changes between the two columns. Differences in color, texture, background,
or style are semantic content the classifier was ignoring. The subject and class-relevant
structure tend to be preserved; everything else is fair game for the null space to absorb.


In [ ]:
#---------------------------- display results --------------------------------#
fig, axes = plt.subplots(len(results) + 1, 2, figsize=(8, 4 * (len(results) + 1)))

axes[0, 0].imshow(image)
axes[0, 0].set_title("input image")
axes[0, 0].axis("off")
axes[0, 1].axis("off")

for row, result in enumerate(results, start=1):
    axes[row, 0].imshow(Image.open(result.original_path))
    axes[row, 0].set_title(f"seed={result.seed} | original reconstruction")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(Image.open(result.principal_path))
    axes[row, 1].set_title(f"seed={result.seed} | equivalent (null-space removed)")
    axes[row, 1].axis("off")

plot_path = repo_root / "outputs" / "notebook_generation_manual.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"saved plot: {plot_path}")
plt.show()

### Part A Recap

The manual pipeline above made each stage explicit: feature extraction, SVD decomposition,
null-space projection, translation, and generation. The math is the same every time you
run it.

Part B shows how the same result is produced through the `SingleImageAnalyzer` API, which
wraps all five steps into a single call. Use Part A to build intuition; use Part B when
you want to run this on your own images quickly.


---
## Part B — High-level API via `SingleImageAnalyzer`

The `SingleImageAnalyzer` runs the exact same pipeline in one call — feature extraction, SVD, translation, and generation — and additionally computes IS/AS scores and dictionary similarity.

### If you previously ran Part A

In [ ]:
#---------------------------- cleanup before part B --------------------------------#
del unclip
import gc
gc.collect()
torch.cuda.empty_cache()
print("VRAM freed")


### If you did not initiate Part A, you may start from here

In [ ]:
#---------------------------- run via SingleImageAnalyzer --------------------------------#
from sing.core.analyzer import SingleImageAnalyzer

analyzer = SingleImageAnalyzer(repo_root=repo_root, device=str(device))

output = analyzer.analyze(
    image_path=image_path,
    model_name="resnet",
    seeds=[42, 1337],
    output_dir=repo_root / "outputs" / "notebook_generation_highlevel",
)

print(f"model={output.model_name}  translator={output.translator_name}")
print(f"IS={output.is_value:.6f}")
if output.as_value is not None:
    print(f"AS={output.as_value:.6f}")
print(f"generated {len(output.generated_files)} seed pair(s)")
for r in output.generated_files:
    print(f"  seed={r.seed}  original={r.original_path.name}  principal={r.principal_path.name}")

In [ ]:
#---------------------------- display results --------------------------------#
fig, axes = plt.subplots(len(output.generated_files) + 1, 2, figsize=(8, 4 * (len(output.generated_files) + 1)))

axes[0, 0].imshow(image)
axes[0, 0].set_title("input image")
axes[0, 0].axis("off")
axes[0, 1].axis("off")

for row, result in enumerate(output.generated_files, start=1):
    axes[row, 0].imshow(Image.open(result.original_path))
    axes[row, 0].set_title(f"seed={result.seed} | original reconstruction")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(Image.open(result.principal_path))
    axes[row, 1].set_title(f"seed={result.seed} | equivalent (null-space removed)")
    axes[row, 1].axis("off")

plot_path = repo_root / "outputs" / "notebook_generation_highlevel.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"saved plot: {plot_path}")
plt.show()